# Borealis Fine-Tuning

Простой notebook для дообучения модели Borealis на своих данных.

**Требования:**
- GPU с 24GB+ VRAM (A100/A10/L4/RTX 4090)
- Датасет в формате HuggingFace с колонками: `audio`, `question`, `answer`

## 1. Установка зависимостей

In [ ]:
!pip install -q torch transformers datasets accelerate tqdm jiwer wandb
!pip install -q liger-kernel

In [ ]:
# Клонируем репозиторий Borealis
!git clone https://github.com/VikhrModels/Borealis.git
%cd Borealis

## 2. Конфигурация

### Пример: Instruct-датасет Speech-Instructions

In [ ]:
# ========== НАСТРОЙКИ ==========

# Датасет для обучения (HuggingFace Hub или локальный путь)
# Примеры датасетов:
#   - "Vikhrmodels/Speech-Instructions" — QA на русском
#   - "Vikhrmodels/ToneBooks" — ASR аудиокниги
#   - "Vikhrmodels/AudioBooksInstructGemini2.5" — instruct на аудиокнигах

TRAIN_DATASET = "Vikhrmodels/Speech-Instructions"
TRAIN_SPLIT = "train"

# Колонки в датасете
AUDIO_COLUMN = "audio"
QUESTION_COLUMN = "question"  # или None для ASR (только транскрипция)
ANSWER_COLUMN = "answer"      # или "text" для ASR

# Количество примеров (None = весь датасет)
MAX_SAMPLES = 5000

# Режим обучения
FULL_FINETUNING = False  # True = обучаем всю модель, False = только adapter (быстрее, меньше памяти)

# Гиперпараметры
BATCH_SIZE = 2
GRADIENT_ACCUMULATION = 8
LEARNING_RATE = 2e-5
NUM_EPOCHS = 1
WARMUP_RATIO = 0.03

# Куда сохранять
OUTPUT_DIR = "./borealis_finetuned"

# W&B (опционально)
USE_WANDB = False
WANDB_PROJECT = "borealis-finetune"

## 3. Загрузка модели и данных

In [ ]:
import os
os.environ["HF_AUDIO_DECODER_BACKEND"] = "soundfile"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
from datasets import load_dataset, Audio
from transformers import (
    AutoTokenizer,
    Qwen3ForCausalLM,
    WhisperModel,
    WhisperFeatureExtractor,
    Trainer,
    TrainingArguments,
)
from liger_kernel.transformers import apply_liger_kernel_to_qwen2

# Оптимизации
apply_liger_kernel_to_qwen2()
torch.backends.cudnn.benchmark = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Загрузка датасета
print(f"Loading dataset: {TRAIN_DATASET}...")
dataset = load_dataset(TRAIN_DATASET, split=TRAIN_SPLIT)

if MAX_SAMPLES:
    dataset = dataset.select(range(min(MAX_SAMPLES, len(dataset))))

# Конвертация аудио в 16kHz
dataset = dataset.cast_column(AUDIO_COLUMN, Audio(sampling_rate=16000))

print(f"Dataset size: {len(dataset)} samples")
print(f"Columns: {dataset.column_names}")

# Посмотрим на примеры
print("\n--- Примеры из датасета ---")
for i in range(min(3, len(dataset))):
    sample = dataset[i]
    print(f"\n[{i}] Question: {sample.get(QUESTION_COLUMN, 'N/A')[:100]}...")
    print(f"    Answer: {sample[ANSWER_COLUMN][:100]}...")

In [ ]:
# Загрузка компонентов модели
print("Loading Whisper encoder...")
whisper_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-large-v3")
audio_encoder = WhisperModel.from_pretrained(
    "openai/whisper-large-v3",
    torch_dtype=torch.bfloat16
).encoder

print("Loading Qwen3-4B...")
language_model = Qwen3ForCausalLM.from_pretrained(
    "Qwen/Qwen3-4B",
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",
)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B", trust_remote_code=True)
tokenizer.add_special_tokens({"additional_special_tokens": ["<|start_of_audio|>", "<|end_of_audio|>"]})

In [ ]:
# Создание модели Borealis
from borealis.modeling import BorealisForConditionalGeneration

print("Creating Borealis model...")
model = BorealisForConditionalGeneration(
    audio_encoder=audio_encoder,
    language_model=language_model,
    tokenizer=tokenizer
)

# Загружаем веса претрейна
print("Loading pretrained weights from HuggingFace...")
from huggingface_hub import hf_hub_download

ckpt_path = hf_hub_download(
    repo_id="Vikhrmodels/Borealis-5b-it",
    filename="pytorch_model.bin"
)
state_dict = torch.load(ckpt_path, map_location="cpu", weights_only=False)
model.load_state_dict(state_dict, strict=False)
del state_dict

# Заморозка LLM если adapter-only режим
if not FULL_FINETUNING:
    print("Freezing LLM, training only adapter...")
    for p in model.llm.parameters():
        p.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## 4. Подготовка данных

In [ ]:
from borealis.dataset import BorealisInstructDataset
from borealis.utils import AudioCollator

# Определяем static_question для ASR режима
static_question = None
q_col = QUESTION_COLUMN
if QUESTION_COLUMN is None:
    static_question = "Транскрибируй аудио."
    q_col = None

train_dataset = BorealisInstructDataset(
    hf_dataset=dataset,
    tokenizer=tokenizer,
    feature_extractor=whisper_extractor,
    max_text_len=1024,
    default_system_prompt="You are a helpful voice assistant.",
    question_column=q_col,
    answer_column=ANSWER_COLUMN,
    audio_column=AUDIO_COLUMN,
    static_question=static_question,
)

collator = AudioCollator()

print(f"Training dataset ready: {len(train_dataset)} samples")

## 5. Обучение

In [ ]:
# Настройки W&B
if USE_WANDB:
    import wandb
    wandb.init(project=WANDB_PROJECT)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    warmup_ratio=WARMUP_RATIO,
    bf16=True,
    logging_steps=10,
    save_steps=500,
    save_total_limit=2,
    dataloader_num_workers=4,
    report_to="wandb" if USE_WANDB else "none",
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    gradient_checkpointing=False,
    save_safetensors=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collator,
)

print("Starting training...")
print(f"  Samples: {len(train_dataset)}")
print(f"  Batch size: {BATCH_SIZE} x {GRADIENT_ACCUMULATION} = {BATCH_SIZE * GRADIENT_ACCUMULATION}")
print(f"  Steps per epoch: {len(train_dataset) // (BATCH_SIZE * GRADIENT_ACCUMULATION)}")

In [ ]:
trainer.train()

In [ ]:
# Сохранение финальной модели
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Saving model to {OUTPUT_DIR}...")
torch.save(model.state_dict(), f"{OUTPUT_DIR}/pytorch_model.bin")
print("Done!")

## 6. Тестирование на примерах из датасета

In [ ]:
# Тест на нескольких примерах из датасета
model.eval()
model = model.to(DEVICE)

import random

print("=" * 60)
print("ТЕСТИРОВАНИЕ НА ПРИМЕРАХ ИЗ ДАТАСЕТА")
print("=" * 60)

# Берём 5 случайных примеров
test_indices = random.sample(range(len(dataset)), min(5, len(dataset)))

for idx in test_indices:
    sample = dataset[idx]
    
    audio = sample[AUDIO_COLUMN]["array"]
    question = sample.get(QUESTION_COLUMN, "Что слышно на аудио?")
    reference = sample[ANSWER_COLUMN]
    
    # Обработка аудио
    mel = whisper_extractor(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    ).input_features.to(DEVICE, dtype=torch.bfloat16)
    
    # Генерация
    with torch.inference_mode():
        output = model.generate(
            mel=[[mel[0]]],
            user_prompt=question,
            max_new_tokens=256,
            do_sample=False,
        )
    
    response = tokenizer.decode(output[0], skip_special_tokens=True)
    
    print(f"\n--- Пример {idx} ---")
    print(f"Question: {question[:150]}{'...' if len(question) > 150 else ''}")
    print(f"Reference: {reference[:200]}{'...' if len(reference) > 200 else ''}")
    print(f"Generated: {response[:200]}{'...' if len(response) > 200 else ''}")
    print("-" * 40)

## 7. Инференс на своём аудио

In [ ]:
# Загрузка и инференс на произвольном аудио
import torchaudio

def transcribe_audio(audio_path, question="Что слышно на аудио?"):
    """Инференс на произвольном аудиофайле."""
    # Загрузка аудио
    waveform, sr = torchaudio.load(audio_path)
    
    # Ресемплинг в 16kHz если нужно
    if sr != 16000:
        waveform = torchaudio.functional.resample(waveform, sr, 16000)
    
    # Моно
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    
    audio = waveform.squeeze().numpy()
    
    # Обработка
    mel = whisper_extractor(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    ).input_features.to(DEVICE, dtype=torch.bfloat16)
    
    # Генерация
    with torch.inference_mode():
        output = model.generate(
            mel=[[mel[0]]],
            user_prompt=question,
            max_new_tokens=256,
            do_sample=False,
        )
    
    return tokenizer.decode(output[0], skip_special_tokens=True)

# Пример использования:
# result = transcribe_audio("path/to/audio.wav", "О чём говорится в аудио?")
# print(result)

## 8. Push to HuggingFace (опционально)

In [ ]:
# Раскомментируйте для загрузки на HuggingFace Hub

# from huggingface_hub import HfApi, login
# 
# login()  # Введите свой токен
# 
# REPO_ID = "YOUR_USERNAME/borealis-finetuned"  # Замените на свой
# 
# api = HfApi()
# 
# # Создаём репозиторий если его нет
# api.create_repo(repo_id=REPO_ID, exist_ok=True)
# 
# # Загружаем веса
# api.upload_file(
#     path_or_fileobj=f"{OUTPUT_DIR}/pytorch_model.bin",
#     path_in_repo="pytorch_model.bin",
#     repo_id=REPO_ID,
#     repo_type="model",
# )
# 
# print(f"Model uploaded to: https://huggingface.co/{REPO_ID}")

---

## Примеры конфигураций для разных задач

### ASR (транскрипция аудиокниг)
```python
TRAIN_DATASET = "Vikhrmodels/ToneBooks"
TRAIN_SPLIT = "train"
AUDIO_COLUMN = "audio"
QUESTION_COLUMN = None  # ASR режим
ANSWER_COLUMN = "text"
```

### Instruct (вопрос-ответ)
```python
TRAIN_DATASET = "Vikhrmodels/Speech-Instructions"
TRAIN_SPLIT = "train"
AUDIO_COLUMN = "audio"
QUESTION_COLUMN = "question"
ANSWER_COLUMN = "answer"
```

### Audio Description (описание аудио)
```python
TRAIN_DATASET = "Vikhrmodels/Speech-Describe"
TRAIN_SPLIT = "500k_part1_speech"
AUDIO_COLUMN = "audio"
QUESTION_COLUMN = "question"
ANSWER_COLUMN = "answer"
```